# 01. DATA INTEGRATION - THU THẬP VÀ TÍCH HỢP DỮ LIỆU
 Data Preparation (Chuẩn bị dữ liệu)

**Mục tiêu:** 
1. Tải tập dữ liệu gốc từ Kaggle.
2. Hợp nhất dữ liệu lịch sử (2009-2023) và dữ liệu mới (2025).
3. Làm giàu dữ liệu bằng cách truy vấn 12 thuộc tính âm học (Audio Features) từ Spotify API.

In [5]:
import pandas as pd
import kagglehub
import os
import shutil
import spotipy
import glob
from spotipy.oauth2 import SpotifyClientCredentials
import time
import warnings
from pathlib import Path
from dotenv import load_dotenv

warnings.filterwarnings('ignore')
print("--- Thư viện đã sẵn sàng ---")

--- Thư viện đã sẵn sàng ---


## 1. Tải dữ liệu từ Kaggle

In [6]:
# Tải dataset (Kiểm tra file đã tồn tại)
raw_folder = "../data/raw/"
os.makedirs(raw_folder, exist_ok=True)

# Kiểm tra file CSV đã tồn tại chưa
csv_files = [f for f in os.listdir(raw_folder) if f.endswith(".csv")]

if csv_files:
    print("✅ Phát hiện file CSV đã tồn tại, khỏi tải từ Kaggle")
    print(f"📋 File tìm thấy:")
    for f in csv_files:
        file_path = os.path.join(raw_folder, f)
        file_size = os.path.getsize(file_path) / (1024 * 1024)  # MB
        print(f"   • {f} ({file_size:.2f} MB)")
else:
    print("📥 Tải dataset từ Kaggle...")
    path = kagglehub.dataset_download("wardabilal/spotify-global-music-dataset-20092025")
    print(f"Dataset đã tải về tại: {path}")
    
    for file in os.listdir(path):
        if file.endswith(".csv"):
            shutil.copy(os.path.join(path, file), raw_folder)
            print(f"✅ Đã sao chép: {file}")

✅ Phát hiện file CSV đã tồn tại, khỏi tải từ Kaggle
📋 File tìm thấy:
   • spotify_data clean.csv (1.35 MB)
   • track_data_final.csv (1.39 MB)


## 2. Hợp nhất các tập dữ liệu gốc

In [7]:
# Đọc dữ liệu
df_2025 = pd.read_csv('../data/raw/spotify_data clean.csv')
df_history = pd.read_csv('../data/raw/track_data_final.csv')

# Gộp thô
df_merged = pd.concat([df_2025, df_history], ignore_index=True)

# Lọc các track_id duy nhất để tối ưu hóa việc gọi API
unique_tracks = df_merged.drop_duplicates(subset=['track_id'])
print(f"Số lượng bản ghi sau khi gộp: {len(df_merged)}")
print(f"Số lượng ID duy nhất cần truy vấn API: {len(unique_tracks)}")

Số lượng bản ghi sau khi gộp: 17360
Số lượng ID duy nhất cần truy vấn API: 8778


## Bổ sung dữ liệu thuộc tính âm học
Thu thập thêm 4 nguồn dữ liệu có thuộc tính âm học, kiểm tra trùng khớp track_id rồi merge dữ liệu

Lưu vào file spotify_merged_with_features_v2.csv

In [9]:
# Hàm FIX CSV ERRORS - Xử lý dòng lỗi (unclosed quotes)
def fix_and_read_csv(file_path, max_attempts=3):
    """
    Đọc file CSV với xử lý lỗi từng dòng.
    Nếu gặp lỗi, tự động fix dấu ngoặc kép và thử lại.
    """
    import csv
    import io
    
    for attempt in range(max_attempts):
        try:
            # Thử 1: Đọc bình thường
            if attempt == 0:
                return pd.read_csv(file_path)
            
            # Thử 2: Skip dòng lỗi
            if attempt == 1:
                return pd.read_csv(file_path, on_bad_lines='skip', engine='python')
            
            # Thử 3: Fix dòng lỗi (thêm dấu ngoặc kép) rồi đọc
            if attempt == 2:
                with open(file_path, 'r', encoding='utf-8', errors='replace') as f:
                    lines = f.readlines()
                
                fixed_lines = [lines[0]]  # Giữ header
                fixed_count = 0
                
                for line in lines[1:]:
                    line = line.rstrip('\n\r')
                    quote_count = line.count('"')
                    
                    # Nếu lẻ số dấu ngoặc kép, thêm một cái ở cuối
                    if quote_count % 2 != 0:
                        line = line + '"'
                        fixed_count += 1
                    
                    fixed_lines.append(line + '\n')
                
                content = ''.join(fixed_lines)
                df = pd.read_csv(io.StringIO(content), on_bad_lines='skip', engine='python')
                print(f"      ✅ Fixed {fixed_count} dòng lỗi (unclosed quotes)")
                return df
        
        except Exception as e:
            if attempt == max_attempts - 1:
                raise e
            continue
    
    return pd.DataFrame()

# 🔍 KIỂM TRA FILE ĐÃ TẠO CHƯA - NẾU CÓ THÌ BỎ QUA
output_check = '../data/processed/spotify_merged_with_features_v2.csv'
if os.path.exists(output_check):
    print("✅ Phát hiện file đã tạo từ lần trước!")
    print(f"📋 File: {output_check}")
    df_check = pd.read_csv(output_check)
    print(f"📊 Dữ liệu: {len(df_check):,} bài hát")
    print("⏭️ BỎ QUA cell này để không tải lại từ Kaggle")
else:
    print("🚀 Lần đầu chạy - Bắt đầu tải từ 4 nguồn Kaggle...\n")
    # --- BƯỚC 1: THU THẬP TỪ 4 NGUỒN ---

    # Nguồn 1: Kaggle - Spotify 1.2M Songs
    print("Đang tải Nguồn 1 (Kaggle 1.2M Songs)...")
    try:
        path_1 = kagglehub.dataset_download("rodolfofigueroa/spotify-12m-songs")
        df1 = pd.read_csv(f"{path_1}/tracks_features.csv")
        if 'id' in df1.columns: df1 = df1.rename(columns={'id': 'track_id'})
        print(f"Nguồn 1: {len(df1):,} bài")
    except Exception as e:
        print(f"Lỗi Nguồn 1: {e}"); df1 = pd.DataFrame()

    # Nguồn 2: Kaggle - Weekly Updated
    print("Đang tải Nguồn 2 (Kaggle Weekly Updated)...")
    try:
        path_2 = kagglehub.dataset_download("gauthamvijayaraj/spotify-tracks-dataset-updated-every-week")
        csv_files_2 = glob.glob(f"{path_2}/*.csv")
        df2 = pd.read_csv(csv_files_2[0]) if csv_files_2 else pd.DataFrame()
        if 'id' in df2.columns: df2 = df2.rename(columns={'id': 'track_id'})
        print(f"Nguồn 2: {len(df2):,} bài")
    except Exception as e:
        print(f"Lỗi Nguồn 2: {e}"); df2 = pd.DataFrame()

    # Nguồn 3: Kaggle - Solomon Ameh (Quét tất cả CSV)
    print("Đang tải Nguồn 3 (Kaggle - Yashdev01)...")
    try:
        path_3 = kagglehub.dataset_download("yashdev01/spotify-tracks-dataset")
        csv_files_3 = glob.glob(f"{path_3}/*.csv")
        df3_list = []
        for file in csv_files_3:
            temp_df = pd.read_csv(file)
            if 'danceability' in temp_df.columns:
                if 'id' in temp_df.columns and 'track_id' not in temp_df.columns:
                    temp_df = temp_df.rename(columns={'id': 'track_id'})
                df3_list.append(temp_df)
        df3 = pd.concat(df3_list, ignore_index=True) if df3_list else pd.DataFrame()
        print(f"Nguồn 3: {len(df3):,} bài")
    except Exception as e:
        print(f"Lỗi Nguồn 3: {e}"); df3 = pd.DataFrame()

    # Nguồn 4: Kaggle - serkantysz (TỰ ĐỘNG FIX DÒNG LỖI EOF)
    print("Đang tải Nguồn 4 (Kaggle - serkantysz)...")
    try:
        path_4 = kagglehub.dataset_download("serkantysz/550k-spotify-songs-audio-lyrics-and-genres")
        csv_files_4 = glob.glob(f"{path_4}/*.csv")
        df4_list = []
        
        for file in csv_files_4:
            print(f"   📄 Xử lý: {os.path.basename(file)}")
            try:
                # Dùng hàm fix_and_read_csv để tự động xử lý lỗi
                temp_df = fix_and_read_csv(file)
                print(f"      ✅ Tải thành công: {len(temp_df):,} bài")
                
                # Kiểm tra xem có audio features không
                if not temp_df.empty and 'danceability' in temp_df.columns:
                    rows_with_audio = temp_df['danceability'].notna().sum()
                    print(f"      📊 Có {rows_with_audio:,} bài với audio features")
                    
                    # Rename 'id' to 'track_id' if needed
                    if 'id' in temp_df.columns and 'track_id' not in temp_df.columns:
                        temp_df = temp_df.rename(columns={'id': 'track_id'})
                    df4_list.append(temp_df)
                else:
                    print(f"      ⚠️  Không có cột danceability hoặc file trống")
                    
            except Exception as file_err:
                print(f"      ❌ Lỗi xử lý file: {str(file_err)[:80]}")
        
        df4 = pd.concat(df4_list, ignore_index=True) if df4_list else pd.DataFrame()
        rows_with_features = df4['danceability'].notna().sum() if not df4.empty else 0
        print(f"Nguồn 4: {len(df4):,} bài, trong đó {rows_with_features:,} bài có audio features")
        
    except Exception as e:
        print(f"Lỗi Nguồn 4: {e}")
        df4 = pd.DataFrame()

    # --- BƯỚC 2: GỘP 4 NGUỒN VÀ LOẠI BỎ TRÙNG LẶP ---

    audio_cols = [
        'track_id', 'danceability', 'energy', 'key', 'loudness', 'mode', 
        'speechiness', 'acousticness', 'instrumentalness', 'liveness', 
        'valence', 'tempo', 'time_signature'
    ]

    print("\nĐang xử lý và gộp 4 siêu kho...")
    pool_list = []
    for df, name in zip([df1, df2, df3, df4], ["Nguồn 1", "Nguồn 2", "Nguồn 3", "Nguồn 4"]):
        if not df.empty:
            avail_cols = [c for c in audio_cols if c in df.columns]
            if 'track_id' in avail_cols:
                pool_list.append(df[avail_cols])

    # Siêu Kho (Mega Pool) 4 Nguồn
    df_mega_pool = pd.concat(pool_list, ignore_index=True)
    df_mega_pool = df_mega_pool.drop_duplicates(subset=['track_id'])
    print(f"TẠO KHO THÀNH CÔNG: Chứa tổng cộng {len(df_mega_pool):,} features duy nhất.")

    # --- BƯỚC 3: MERGE VÀO DỮ LIỆU GỐC CỦA BẠN ---

    # Dọn dẹp df_merged gốc
    cols_to_drop = [c for c in audio_cols if c != 'track_id' and c in df_merged.columns]
    df_merged_clean = df_merged.drop(columns=cols_to_drop)

    # Hợp nhất
    df_final_real = df_merged_clean.merge(df_mega_pool, on='track_id', how='left')

    # Tính toán kết quả
    total_tracks = len(df_final_real)
    found_tracks = df_final_real['danceability'].notna().sum()
    missing_tracks = total_tracks - found_tracks

    print(f"\nKẾT QUẢ MERGE CUỐI CÙNG (4 NGUỒN):")
    print(f"   - Tổng số bài hát của bạn: {total_tracks:,}")
    print(f"   - Số bài TÌM THẤY dữ liệu thật: {found_tracks:,} ({(found_tracks/total_tracks)*100:.2f}%)")
    print(f"   - Số bài còn thiếu: {missing_tracks:,}")

    # Lưu file hoàn chỉnh phiên bản V2
    os.makedirs('../data/processed', exist_ok=True)
    output_path = '../data/processed/spotify_merged_with_features_v2.csv'
    df_final_real.to_csv(output_path, index=False)
    print(f"Đã lưu file dữ liệu thực tế tại: {output_path}")

🚀 Lần đầu chạy - Bắt đầu tải từ 4 nguồn Kaggle...

Đang tải Nguồn 1 (Kaggle 1.2M Songs)...
Nguồn 1: 1,204,025 bài
Đang tải Nguồn 2 (Kaggle Weekly Updated)...
Nguồn 2: 62,317 bài
Đang tải Nguồn 3 (Kaggle - Yashdev01)...
Nguồn 3: 114,000 bài
Đang tải Nguồn 4 (Kaggle - serkantysz)...
   📄 Xử lý: artists.csv
      ✅ Tải thành công: 71,440 bài
      ⚠️  Không có cột danceability hoặc file trống
   📄 Xử lý: songs.csv
      ✅ Tải thành công: 550,622 bài
      📊 Có 550,622 bài với audio features
Nguồn 4: 550,622 bài, trong đó 550,622 bài có audio features

Đang xử lý và gộp 4 siêu kho...
TẠO KHO THÀNH CÔNG: Chứa tổng cộng 1,690,027 features duy nhất.

KẾT QUẢ MERGE CUỐI CÙNG (4 NGUỒN):
   - Tổng số bài hát của bạn: 17,360
   - Số bài TÌM THẤY dữ liệu thật: 7,337 (42.26%)
   - Số bài còn thiếu: 10,023
Đã lưu file dữ liệu thực tế tại: ../data/processed/spotify_merged_with_features_v2.csv


## Điền khuyết dữ liệu âm học
Kiểm tra những bài hát chưa có dữ liệu của thuộc tính âm học, lấy tên tác giả, tìm kiếm các bài hát khác của tác giả đó đã có dữ liệu, sau đó tính trung bình và điền vào những bài còn thiếu.

Lưu vào file spotify_final_imputed.csv -> Sử dụng file này cho bước cleaning

In [10]:
# 1. Đọc dữ liệu từ file bạn vừa merge thành công
file_path = '../data/processed/spotify_merged_with_features_v2.csv'
df = pd.read_csv(file_path)

print(f"Kích thước ban đầu: {df.shape}")

# 2. TÍNH TOÁN track_duration_ms từ track_duration_min
if 'track_duration_min' in df.columns:
    df['track_duration_ms'] = (df['track_duration_min'] * 60 * 1000).round(0)
    print(f"✅ Đã tính track_duration_ms từ track_duration_min")
elif 'duration_ms' in df.columns and 'track_duration_ms' not in df.columns:
    df['track_duration_ms'] = df['duration_ms']
    print(f"✅ Đã lấy track_duration_ms từ cột duration_ms")

# 3. Định nghĩa các cột đặc trưng âm học cần điền khuyết
audio_features = [
    'danceability', 'energy', 'key', 'loudness', 'mode', 
    'speechiness', 'acousticness', 'instrumentalness', 'liveness', 
    'valence', 'tempo'
]

# ⚠️ LƯU Ý: Thay đổi 'artist_name' thành tên cột chứa tên nghệ sĩ thực tế trong file của bạn
# (Có thể là 'artist', 'artists', 'track_artist'...)
artist_col = 'artist_name' 

print("\n--- BẮT ĐẦU ĐIỀN KHUYẾT THÔNG MINH ---")
print(f"Số lượng NaN (thiếu) trước khi xử lý:\n{df[audio_features].isna().sum()}\n")

# BƯỚC 1: Điền khuyết bằng giá trị TRUNG BÌNH của TỪNG NGHỆ SĨ
# transform('mean') sẽ tính trung bình cho từng nhóm nghệ sĩ và gán lại đúng kích thước cột
for feature in audio_features:
    if feature in df.columns:
        artist_means = df.groupby(artist_col)[feature].transform('mean')
        df[feature] = df[feature].fillna(artist_means)

print("Đã xong Bước 1: Điền dữ liệu dựa trên trung bình của nghệ sĩ.")

# BƯỚC 2: Điền khuyết bằng giá trị TRUNG VỊ của TOÀN BỘ TẬP DỮ LIỆU
# Xử lý những nghệ sĩ hoàn toàn mới, không có bài nào trong tập chuẩn
# for feature in audio_features:
#     if feature in df.columns:
#         global_median = df[feature].median()
#         df[feature] = df[feature].fillna(global_median)

# print("Đã xong Bước 2: Xử lý các nghệ sĩ mới bằng trung vị toàn cục.")

# 3. Kiểm tra lại xem còn sót NaN nào không
print(f"\nSố lượng NaN sau khi xử lý :\n{df[audio_features].isna().sum()}\n")

# 4. Lưu lại bộ dữ liệu đã hoàn thiện
output_path = '../data/processed/spotify_final_imputed.csv'
df.to_csv(output_path, index=False)
print(f"✅ Đã lưu bộ dữ liệu SẠCH 100% tại: {output_path}")
print(f"Shape cuối cùng: {df.shape}")

Kích thước ban đầu: (17360, 28)
✅ Đã tính track_duration_ms từ track_duration_min

--- BẮT ĐẦU ĐIỀN KHUYẾT THÔNG MINH ---
Số lượng NaN (thiếu) trước khi xử lý:
danceability        10023
energy              10023
key                 10023
loudness            10023
mode                10023
speechiness         10023
acousticness        10023
instrumentalness    10023
liveness            10023
valence             10023
tempo               10023
dtype: int64

Đã xong Bước 1: Điền dữ liệu dựa trên trung bình của nghệ sĩ.

Số lượng NaN sau khi xử lý :
danceability        4884
energy              4884
key                 4884
loudness            4884
mode                4884
speechiness         4884
acousticness        4884
instrumentalness    4884
liveness            4884
valence             4884
tempo               4884
dtype: int64

✅ Đã lưu bộ dữ liệu SẠCH 100% tại: ../data/processed/spotify_final_imputed.csv
Shape cuối cùng: (17360, 28)


In [11]:
# ===== DIAGNOSTIC: KIỂM TRA CHI TIẾT DỮ LIỆU MERGE TỪ 4 NGUỒN =====
print("="*80)
print("🔍 PHÂN TÍCH CHI TIẾT DỮ LIỆU MERGE TỪ 4 NGUỒN")
print("="*80)

# Đọc file merged
merged_file = '../data/processed/spotify_merged_with_features_v2.csv'
df_merged = pd.read_csv(merged_file)

print(f"\n📊 TỔNG QUAN FILE MERGED:")
print(f"   Shape: {df_merged.shape}")
print(f"   Số cột: {len(df_merged.columns)}")
print(f"   Số dòng: {len(df_merged):,}")

# Kiểm tra các cột có sẵn
print(f"\n📋 DANH SÁCH TẤT CẢ CỘT:")
for i, col in enumerate(df_merged.columns, 1):
    print(f"   {i:2d}. {col}")

# Kiểm tra các cột audio features
audio_cols_needed = [
    'danceability', 'energy', 'key', 'loudness', 'mode', 
    'speechiness', 'acousticness', 'instrumentalness', 'liveness', 
    'valence', 'tempo', 'time_signature'
]

print(f"\n🎵 PHÂN TÍCH CỘT AUDIO FEATURES:")
print(f"   Các cột cần có: {audio_cols_needed}")

available_audio = [col for col in audio_cols_needed if col in df_merged.columns]
missing_audio = [col for col in audio_cols_needed if col not in df_merged.columns]

print(f"\n   ✅ Cột CÓ: {len(available_audio)}")
for col in available_audio:
    non_null = df_merged[col].notna().sum()
    null_pct = (df_merged[col].isna().sum() / len(df_merged)) * 100
    print(f"      • {col}: {non_null:,} non-null ({100-null_pct:.1f}%), {df_merged[col].isna().sum():,} NaN ({null_pct:.1f}%)")

print(f"\n   ❌ Cột THIẾU: {len(missing_audio)}")
for col in missing_audio:
    print(f"      • {col}")

# Kiểm tra những bài hát nào có đầy đủ audio features
print(f"\n📈 PHÂN TÍCH TÍNH ĐẦY ĐỦ CỦA DỮ LIỆU:")
core_features = ['danceability', 'energy', 'tempo']  # 3 cột quan trọng nhất
complete_rows = df_merged[core_features].notna().all(axis=1).sum()
incomplete_rows = len(df_merged) - complete_rows

print(f"   Bài hát có đầy đủ {core_features}: {complete_rows:,} ({(complete_rows/len(df_merged))*100:.2f}%)")
print(f"   Bài hát THIẾU bất kỳ: {incomplete_rows:,} ({(incomplete_rows/len(df_merged))*100:.2f}%)")

# Chi tiết từng nguồn - kiểm tra xem có bao nhiêu bài từ mỗi nguồn
print(f"\n🔎 KIỂM TRA NGUỒN DỮ LIỆU (dựa trên cột 'time_signature'):")
if 'time_signature' in df_merged.columns:
    ts_coverage = df_merged['time_signature'].notna().sum()
    print(f"   Bài hát có time_signature (dấu hiệu từ Kaggle): {ts_coverage:,}")
    print(f"   Bài hát KHÔNG có: {len(df_merged) - ts_coverage:,}")

# Kiểm tra phân phối các thuộc tính
print(f"\n📊 THỐNG KÊ CỘT AUDIO FEATURES CÓ SẴN:")
for col in available_audio:
    if df_merged[col].notna().sum() > 0:
        valid_data = df_merged[col].dropna()
        print(f"   {col}:")
        print(f"      Min: {valid_data.min():.4f}, Max: {valid_data.max():.4f}, Mean: {valid_data.mean():.4f}")

# Kiểm tra 5 dòng đầu của file merged để xem dữ liệu có gì
print(f"\n👀 5 DÒNG ĐẦU TIÊN CỦA FILE MERGED:")
print(df_merged[['track_id', 'track_name', 'artist_name'] + available_audio[:3]].head(10))

print("\n" + "="*80)

🔍 PHÂN TÍCH CHI TIẾT DỮ LIỆU MERGE TỪ 4 NGUỒN

📊 TỔNG QUAN FILE MERGED:
   Shape: (17360, 28)
   Số cột: 28
   Số dòng: 17,360

📋 DANH SÁCH TẤT CẢ CỘT:
    1. track_id
    2. track_name
    3. track_number
    4. track_popularity
    5. explicit
    6. artist_name
    7. artist_popularity
    8. artist_followers
    9. artist_genres
   10. album_id
   11. album_name
   12. album_release_date
   13. album_total_tracks
   14. album_type
   15. track_duration_min
   16. track_duration_ms
   17. danceability
   18. energy
   19. key
   20. loudness
   21. mode
   22. speechiness
   23. acousticness
   24. instrumentalness
   25. liveness
   26. valence
   27. tempo
   28. time_signature

🎵 PHÂN TÍCH CỘT AUDIO FEATURES:
   Các cột cần có: ['danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature']

   ✅ Cột CÓ: 12
      • danceability: 7,337 non-null (42.3%), 10,023 NaN (57.7%)
      • energy: 7,3